# Часть 4 — один трейс на двух агентов

В части 3 у риск-менеджера и аналитика получились **разные трейсы**. Хочется одно дерево:

```
🤖 risk-manager
   🧠 LLM: «надо спросить аналитика»
   📨 A2A → analyst
      🤖 analyst                  ← шаги ДРУГОГО процесса, внутри нашего трейса
         🧠 LLM → 🔧 run_sql → 🧠 LLM …
   🧠 LLM: решение
```

**Как это сделать.** Аналитику нужно знать, в какой трейс писать. Передадим ему это вместе с
вопросом — в `metadata` A2A-сообщения: номер трейса (`trace_id`) и номер шага-родителя
(`parent_span_id`). Это обычный приём распределённой трассировки (distributed tracing): так же
сервисы передают друг другу заголовок `traceparent`.

**Перед запуском** должны работать (каждое — в своём терминале, из папки занятия):

```bash
docker compose up -d                  # база банка + Langfuse (один раз)
uv run python mcp_server.py           # терминал 1: MCP-сервер с инструментами → :8101
uv run python analyst_agent.py        # терминал 2: агент-аналитик по A2A     → :8102
```

In [1]:
import httpx

for name, url in [("MCP-сервер", "http://localhost:8101/mcp"), ("аналитик", "http://localhost:8102/.well-known/agent-card.json")]:
    try:
        httpx.get(url, timeout=3)
        print(f"✅ {name} работает")
    except httpx.ConnectError:
        print(f"❌ {name} не запущен — см. команды выше")

import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(model=os.getenv("MODEL", "openai/gpt-4o-mini"), base_url="https://openrouter.ai/api/v1",
                 api_key=os.getenv("OPENROUTER_API_KEY"))

from langfuse import get_client, propagate_attributes
from langfuse.langchain import CallbackHandler

from risk_agent import build_risk_agent, send_to_analyst
from trace_view import print_trace

langfuse = get_client()

✅ MCP-сервер работает
✅ аналитик работает


## Шаг 1. Сторона аналитика — уже готова

В `analyst_agent.py` аналитик смотрит в `metadata` входящего сообщения. Есть `trace_id` — продолжает
**чужой** трейс: `caller_span(...)` делает шаг риск-менеджера родителем всех своих шагов (обычный
механизм OpenTelemetry, на котором построен Langfuse). Нет — открывает свой трейс `analyst`:

In [2]:
import inspect

import analyst_agent

print(inspect.getsource(analyst_agent.AnalystExecutor.answer))

    async def answer(self, question: str, metadata: dict) -> str:
        if self.langfuse is None:                            # no tracing
            result = await self.graph.ainvoke({"messages": [("user", question)]})
            return result["messages"][-1].text

        # with tracing (part 3). If the caller sent its trace id (part 4), our steps go INTO its trace.
        from langfuse import propagate_attributes
        from langfuse.langchain import CallbackHandler
        if "trace_id" in metadata and "parent_span_id" in metadata:   # the caller is traced → continue ITS trace
            parent, attributes = caller_span(metadata["trace_id"], metadata["parent_span_id"]), {}
        else:                            # nobody upstream → our own trace, named "analyst"
            parent, attributes = nullcontext(), {"trace_name": "analyst"}
        if "session_id" in metadata:     # the user's session: links our trace to the caller's in Langfuse
            attributes["session_id"

А вот и `caller_span` — пять строк стандартного OpenTelemetry:

In [3]:
print(inspect.getsource(analyst_agent.caller_span))

@contextmanager
def caller_span(trace_id: str, parent_span_id: str):
    """Make a step of ANOTHER process (the caller's span) the parent of what we trace here.

    Standard OpenTelemetry: the ids came with the A2A message, we put them into the
    current context, and Langfuse attaches our spans under that step of that trace.
    """
    parent = otel_trace.NonRecordingSpan(otel_trace.SpanContext(
        trace_id=int(trace_id, 16), span_id=int(parent_span_id, 16),
        is_remote=True, trace_flags=otel_trace.TraceFlags(otel_trace.TraceFlags.SAMPLED)))
    token = otel_context.attach(otel_trace.set_span_in_context(parent))
    try:
        yield
    finally:
        otel_context.detach(token)



## Шаг 2. Сторона риск-менеджера — передаём номер трейса

Новая версия инструмента `ask_analyst`: открываем шаг «📨 A2A → analyst» и кладём в `metadata`
номер трейса и номер этого шага (плюс номер сессии, как и раньше). Всё остальное — как раньше.

In [4]:
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool


@tool
async def ask_analyst(question: str, config: RunnableConfig) -> str:
    """Ask the bank's analyst agent a factual question about the bank data (accounts, transactions,
    balances, loans). Ask concrete questions with the account number and the period."""
    with langfuse.start_as_current_observation(name="A2A → analyst", input=question) as span:
        metadata = {"trace_id": langfuse.get_current_trace_id(),   # which trace
                    "parent_span_id": span.id,                     # under which step
                    "session_id": config["configurable"]["thread_id"]}   # which session (as in part 3)
        print(f"   📨 → аналитику: {question}")
        answer = await send_to_analyst(question, metadata)
        print(f"   📨 ← аналитик: {answer[:200]}")
        span.update(output=answer)
    return answer


risk_manager = build_risk_agent(llm, ask_tool=ask_analyst)   # the same agent, new tool

## Шаг 3. Запускаем

In [5]:
async def decide(question: str, chat: str) -> str:
    with langfuse.start_as_current_observation(name="risk-manager", as_type="agent", input=question) as root, \
            propagate_attributes(trace_name="risk-manager (+ analyst)", session_id=chat, user_id="student"):
        result = await risk_manager.ainvoke({"messages": [("user", question)]},
                                            {"configurable": {"thread_id": chat}, "callbacks": [CallbackHandler()]})
        root.update(output=result["messages"][-1].text)
        trace_id = langfuse.get_current_trace_id()
    print("\n⚖️", result["messages"][-1].text)
    print("\n🔎 трейс:", langfuse.get_trace_url(trace_id=trace_id))
    return trace_id


trace_id = await decide("Можно ли выдать счёту 1583 кредит 300 000 CZK на 36 месяцев?", chat="client-1583")

   📨 → аналитику: Для счёта 1583 укажи средний ежемесячный приход на счёт за 1998 год в CZK (итого приход за год и среднемесячное значение).
   📨 → аналитику: Для счёта 1583 укажи средний баланс за 1998 год в CZK.
   📨 → аналитику: Для счёта 1583 перечисли все кредиты за всё время с их статусами; отдельно укажи, есть ли статус running_in_debt или finished_default.
   📨 ← аналитик: Для счёта **1583** найден 1 кредит:

- **Кредит 5282** — выдан 24.08.1996; сумма **388 512 CZK**; срок **48 месяцев**; ежемесячный платёж **8 094 CZK**; статус **running_in_debt**.

Статус **running_i
   📨 ← аналитик: Счёт 1583, 1998 год:

- **Общий приход:** 839 333 CZK
- **Среднемесячный приход:** 69 944,42 CZK (общая сумма ÷ 12)
   📨 ← аналитик: Средний баланс счёта 1583 за 1998 год: **62 751,87 CZK**.

⚖️ **ОТКАЗАТЬ**

1. **Платёжеспособность:** платёж по новому кредиту — **9 400,91 CZK/мес.** Лимит 30% от среднемесячного прихода (**69 944,42 CZK**) — **20 983,33 CZK/мес.** Условие выполнено.
2. **Кредитн

Одно дерево на два агента и два процесса:

In [ ]:
print_trace(trace_id)

## Как читать такой трейс

- **кто кого вызвал**: риск-менеджер → A2A → аналитик → MCP-инструменты (`list_tables`, `run_sql`…);
  шаги «📨 A2A → analyst» висят прямо под `risk-manager` — по одному на каждый вопрос аналитику;
- **где время**: обычно больше всего — генерации LLM; инструменты и база — доли секунды. Вопросы,
  которые риск-менеджер задал параллельно, идут одновременно;
- **сколько стоит каждый агент**: строка «токены по агентам» — аналитик часто тратит больше, потому
  что каждый раз заново изучает таблицы. Это подсказка для оптимизации: например, отдать ему схему
  базы в системный промпт (как в сервисе занятия 2);
- **где ошибка**: если решение неверное, по трейсу видно, на каком шаге — плохой вопрос
  риск-менеджера, неверный SQL аналитика или ошибка в рассуждении.

В Langfuse UI по ссылке 🔎 то же дерево интерактивное: у каждой генерации — полный промпт и ответ.

## Итоги занятия

- **A2A** — агенты общаются как сервисы: карточка, сообщения, задачи. Внутренности каждого скрыты;
- второй агент вызывает первого как инструмент; у каждого свои знания и права;
- **Langfuse** показывает каждый шаг: LLM, инструменты, токены, время, сессии, оценки;
- **сквозной трейс**: передаём `trace_id` в `metadata` A2A-сообщения — и работа нескольких
  агентов видна одним деревом.